# Notebook 2 — Complaint Prediction from Vitals (no text needed)

## What this proves
When a patient can't describe symptoms — elderly, low literacy, semi-conscious — DiagNXT can still **suggest the most likely complaint category** from vitals + age + gender alone. The PHC nurse just takes the vitals; the model gives the doctor a head start.

## Algorithm choice
- **XGBoost multiclass** (primary) — handles the 21-class imbalanced problem with class weights, native NaN handling, fastest tabular-classification SOTA.
- **Random Forest** (baseline for comparison) — to honestly report whether XGBoost is actually buying us anything on this dataset.
- We report **top-1, top-3, and top-5 accuracy** because the use case is *suggestion*, not deterministic diagnosis.

## Honest expectation
Vitals + demographics carry limited signal — many complaints (Cough, Fever, Headache) have similar vital profiles. Top-1 likely ~25–35%, top-3 ~50–60%. **This is the correct answer to put on the slide**: "vitals alone aren't enough to diagnose, which is *exactly why* DiagNXT couples vitals with voice symptom intake."

## 1. Setup

In [ ]:
# Install dependencies (Colab-ready). Skips silently if already installed.
!pip install -q pandas numpy scikit-learn xgboost lightgbm matplotlib seaborn joblib 

In [ ]:
# Accelerator detection (informational — boosting models are CPU-bound)
import os, platform, subprocess, sys
ACCELERATOR = 'cpu'
DEVICE_NOTE = ''
try:
    import jax
    if any('TPU' in d.device_kind.upper() for d in jax.devices()):
        ACCELERATOR = 'tpu'
        DEVICE_NOTE = 'TPU detected — note: GBM/RF are CPU-bound; TPU unused. Training is fast anyway.'
except Exception:
    pass
if ACCELERATOR == 'cpu':
    try:
        out = subprocess.check_output(['nvidia-smi','-L'], stderr=subprocess.DEVNULL).decode()
        if 'GPU' in out:
            ACCELERATOR = 'gpu'
            DEVICE_NOTE = f'GPU detected: {out.strip().splitlines()[0]}. XGBoost will use tree_method="hist", device="cuda".'
    except Exception:
        pass
if ACCELERATOR == 'cpu':
    DEVICE_NOTE = f'Running on CPU ({platform.processor() or platform.machine()}). Fine — datasets are small.'
print(f'Accelerator: {ACCELERATOR.upper()}')
print(DEVICE_NOTE)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import re, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
pd.set_option('display.max_columns', 60)
sns.set_style('whitegrid')
RANDOM_STATE = 42

## Load CSV (Colab uploads, locally auto-detects)

In [ ]:
IN_COLAB = 'google.colab' in sys.modules
csv_path = None
if IN_COLAB:
    from google.colab import files
    print('Upload ap_hackathon_data- Worksheet.csv:')
    uploaded = files.upload()
    csv_path = Path(next(iter(uploaded.keys())))
else:
    CANDIDATES = [
        Path.home() / 'Downloads' / 'ap_hackathon_data- Worksheet.csv',
        Path('../data/ap_hackathon_data- Worksheet.csv'),
        Path('ap_hackathon_data- Worksheet.csv'),
    ]
    csv_path = next((p for p in CANDIDATES if p.exists()), None)
    assert csv_path is not None, 'CSV not found.'
df = pd.read_csv(csv_path, low_memory=False)
print(f'Loaded: {csv_path}')
print(f'Shape : {df.shape}')

## Data cleaning

The raw CSV has bad vitals values (SpO2 = 0 means 'not measured', systole has int-max garbage, pulse maxes at 821). We clip every vital to a physiological range and treat out-of-range values as NaN so LightGBM/XGBoost handle them as missing.

In [ ]:
VITAL_RANGES = {
    'temperature': (94, 108),     # °F
    'pulse': (30, 220),
    'respiratory_rate': (8, 60),
    'systole': (60, 240),
    'diastole': (30, 160),
    'spo2': (60, 100),
    'rbs': (40, 600),             # mg/dl
    'bmi': (10, 60),
}
for col, (lo, hi) in VITAL_RANGES.items():
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[(df[col] < lo) | (df[col] > hi), col] = np.nan

# Age (DOB string) → years
def to_age_years(s):
    dt = pd.to_datetime(s, errors='coerce')
    if pd.isna(dt): return np.nan
    yrs = (pd.Timestamp('2026-04-30') - dt).days / 365.25
    return float(yrs) if 0 <= yrs <= 110 else np.nan
df['age_years'] = df['age'].apply(to_age_years)

# Severity may contain '1,2' style — keep max
def parse_severity(s):
    if pd.isna(s): return np.nan
    try: return max(int(x) for x in str(s).split(',') if x.strip().isdigit())
    except: return np.nan
df['severity_num'] = df['severity'].apply(parse_severity)

# Duration
df['duration_days'] = pd.to_numeric(df['duration_days'], errors='coerce').clip(0, 365)

print('After cleaning, valid (non-NaN) counts:')
for c in list(VITAL_RANGES) + ['age_years','severity_num','duration_days']:
    print(f'  {c:20s}  {df[c].notna().sum():,}')

## 2. Build the multiclass target (top-20 + 'Other')

505 unique complaints in the raw data, but a long tail. Top-20 covers ~61% of visits; we map everything else to 'Other'.

In [ ]:
TOP_K = 20
top_complaints = df['complaint_name'].value_counts().head(TOP_K).index.tolist()
df['target'] = df['complaint_name'].where(df['complaint_name'].isin(top_complaints), other='Other')
df_train = df.dropna(subset=['target']).copy()
print(f'Classes: {df_train["target"].nunique()}  |  Rows: {len(df_train):,}')
df_train['target'].value_counts().head(22)

## 3. Feature matrix

In [ ]:
NUMERIC = ['age_years','temperature','pulse','respiratory_rate','systole','diastole',
           'spo2','rbs','bmi','duration_days','severity_num']
CATEGORICAL = ['gender']
FEATURES = NUMERIC + CATEGORICAL

X = df_train[FEATURES].copy()
for c in CATEGORICAL:
    X[c] = X[c].astype('category')

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df_train['target'])
print('Feature shape:', X.shape, ' Classes:', len(le.classes_))

## 4. Train/test split (stratified)

In [ ]:
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
print(f'Train: {X_tr.shape}  Test: {X_te.shape}')

## 5. Model A — Random Forest (baseline)
Quick baseline. RF can't handle pandas categoricals natively, so we one-hot encode `gender`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, top_k_accuracy_score, classification_report

X_tr_rf = pd.get_dummies(X_tr, columns=CATEGORICAL, drop_first=False).fillna(-1)
X_te_rf = pd.get_dummies(X_te, columns=CATEGORICAL, drop_first=False).fillna(-1)
X_te_rf = X_te_rf.reindex(columns=X_tr_rf.columns, fill_value=-1)

rf = RandomForestClassifier(n_estimators=300, max_depth=None, min_samples_leaf=5,
                            class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X_tr_rf, y_tr)
rf_proba = rf.predict_proba(X_te_rf)
rf_pred  = rf_proba.argmax(axis=1)

print(f'RF top-1 accuracy: {accuracy_score(y_te, rf_pred)*100:.1f}%')
print(f'RF top-3 accuracy: {top_k_accuracy_score(y_te, rf_proba, k=3, labels=range(len(le.classes_)))*100:.1f}%')
print(f'RF top-5 accuracy: {top_k_accuracy_score(y_te, rf_proba, k=5, labels=range(len(le.classes_)))*100:.1f}%')

## 6. Model B — XGBoost multiclass (primary)

Uses `tree_method='hist'` (fast, low-memory) and `device='cuda'` if a GPU was detected.

In [ ]:
import xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight

sw = compute_sample_weight(class_weight='balanced', y=y_tr)
xgb_kwargs = dict(
    n_estimators=400, max_depth=6, learning_rate=0.1,
    objective='multi:softprob', num_class=len(le.classes_),
    tree_method='hist', random_state=RANDOM_STATE,
    eval_metric='mlogloss',
)
if ACCELERATOR == 'gpu':
    xgb_kwargs['device'] = 'cuda'

xgbc = xgb.XGBClassifier(**xgb_kwargs)
X_tr_x = X_tr.copy(); X_te_x = X_te.copy()
for c in CATEGORICAL:
    X_tr_x[c] = X_tr_x[c].cat.codes.replace(-1, np.nan)
    X_te_x[c] = X_te_x[c].astype('category').cat.codes.replace(-1, np.nan)

xgbc.fit(X_tr_x, y_tr, sample_weight=sw)
xgb_proba = xgbc.predict_proba(X_te_x)
xgb_pred  = xgb_proba.argmax(axis=1)

print(f'XGB top-1 accuracy: {accuracy_score(y_te, xgb_pred)*100:.1f}%')
print(f'XGB top-3 accuracy: {top_k_accuracy_score(y_te, xgb_proba, k=3, labels=range(len(le.classes_)))*100:.1f}%')
print(f'XGB top-5 accuracy: {top_k_accuracy_score(y_te, xgb_proba, k=5, labels=range(len(le.classes_)))*100:.1f}%')

## 7. Side-by-side and top-3 hit chart

In [ ]:
import pandas as pd
results = pd.DataFrame({
    'metric': ['top-1','top-3','top-5'],
    'RandomForest': [
        accuracy_score(y_te, rf_pred),
        top_k_accuracy_score(y_te, rf_proba, k=3, labels=range(len(le.classes_))),
        top_k_accuracy_score(y_te, rf_proba, k=5, labels=range(len(le.classes_))),
    ],
    'XGBoost': [
        accuracy_score(y_te, xgb_pred),
        top_k_accuracy_score(y_te, xgb_proba, k=3, labels=range(len(le.classes_))),
        top_k_accuracy_score(y_te, xgb_proba, k=5, labels=range(len(le.classes_))),
    ],
})
results[['RandomForest','XGBoost']] = (results[['RandomForest','XGBoost']]*100).round(1)
print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(7,3.5))
results.set_index('metric').plot.bar(ax=ax, color=['#9ca3af','#2563eb'], rot=0)
ax.set_ylabel('accuracy %'); ax.set_title('Complaint prediction from vitals — RF vs XGBoost')
ax.set_ylim(0,100); plt.tight_layout(); plt.show()

## 8. Per-class XGBoost report (top complaints)

In [ ]:
print(classification_report(y_te, xgb_pred,
      target_names=le.classes_, zero_division=0))

## 9. Inference function

In [ ]:
BEST = xgbc
BEST_FEATURES = list(X_tr_x.columns)

def predict_complaint(patient: dict, k: int = 3):
    row = {f: patient.get(f, np.nan) for f in NUMERIC}
    g = patient.get('gender', 'F')
    row['gender'] = {'F':0,'M':1}.get(g, np.nan)
    x = pd.DataFrame([row])[BEST_FEATURES]
    p = BEST.predict_proba(x)[0]
    order = np.argsort(-p)[:k]
    return [(le.classes_[i], float(p[i])) for i in order]

case = dict(age_years=58, gender='M', temperature=101, pulse=98, respiratory_rate=22,
            systole=130, diastole=84, spo2=96, rbs=180, bmi=24, duration_days=3, severity_num=2)
print('Demo — adult male, T=101°F, RBS=180, 3-day complaint')
for c,p in predict_complaint(case): print(f'  {p*100:5.1f}%  {c}')

## 10. Save artifacts

In [ ]:
ART = Path('complaint_model'); ART.mkdir(exist_ok=True)
BEST.save_model(str(ART/'xgb_complaint.json'))
joblib.dump({'classes': list(le.classes_), 'features': BEST_FEATURES,
             'numeric': NUMERIC, 'categorical': CATEGORICAL,
             'top_complaints': top_complaints}, ART/'metadata.joblib')
print('Saved:'); [print(' ', p.name, f'{p.stat().st_size/1024:.1f} KB') for p in ART.iterdir()]
if IN_COLAB:
    from google.colab import files
    for p in ART.iterdir(): files.download(str(p))

## 11. Deck framing

- **Use case:** patient walks in unable to articulate symptoms → nurse takes vitals → DiagNXT suggests top-3 likely complaints to the doctor.
- **Result:** XGBoost top-3 ≈ XX% (fill in from cell 7). Vitals carry useful but limited signal.
- **Honest pitch:** *vitals alone aren't enough — that's why DiagNXT combines them with voice symptom intake. The model below is the second column of evidence the doctor sees.*